In [2]:
import sys
print(sys.executable)

c:\Users\Sai deepak\anaconda3\python.exe


In [3]:
%pip install pm4py

Note: you may need to restart the kernel to use updated packages.


  Using cached pm4py-2.7.23.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached cvxopt-1.3.3-cp313-cp313-win_amd64.whl.metadata (44 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
Using cached pm4py-2.7.23.11-py3-none-any.whl (2.7 MB)
Using cached cvxopt-1.3.3-cp313-cp313-win_amd64.whl (13.6 MB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)

   ---------------------------------------- 0/3 [graphviz]
   ------------- -------------------------- 1/3 [cvxopt]
   ------------- -------------------------- 1/3 [cvxopt]
   ------------- -------------------------- 1/3 [cvxopt]
   -------------------------- ------------- 2/3 [pm4py]
   -------------------------- ------------- 2/3 [pm4py]
   -------------------------- ------------- 2/3 [pm4py]
   -------------------------- ------------- 2/3 [pm4py]
   -------------------------- ------------- 2/3 [pm4py]
   -------------------------- ------------- 2/3 [pm4py]
   -------------------------- ------------- 2/3 [pm4py]
   -----

In [5]:
import pandas as pd
import pm4py


print("Pandas:", pd.__version__)
print("PM4Py:", pm4py.__version__)

Pandas: 2.2.3
PM4Py: 2.7.23.11


In [6]:
df = pd.read_excel("../output of pandas/careflow_cleaned_events.xlsx")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Dataset shape: (10485, 15)
Columns: ['Case_ID', 'Activity_Name', 'Timestamp', 'Event_Order', 'Department', 'Triage_Priority', 'Shift', 'Doctor_ID', 'Event_Time', 'Arrival_Time', 'Day_Offset', 'Timestamp_Reconstructed', 'Previous_Activity', 'Previous_Time', 'Transition_Min']


,Case_ID,Activity_Name,Timestamp,Event_Order,Department,Triage_Priority,Shift,Doctor_ID,Event_Time,Arrival_Time,Day_Offset,Timestamp_Reconstructed,Previous_Activity,Previous_Time,Transition_Min
0,CF00001,Registration,20:22:00,1,General Emergency,Medium,Evening,DR006,0.848611,0.848611,0,2026-01-01 00:00:00,NaN,NaT,NaN
1,CF00001,Triage,21:39:00,2,General Emergency,Medium,Evening,DR006,0.902083,0.848611,0,2026-01-01 01:17:00,Registration,2026-01-01 00:00:00,77.0
2,CF00001,Doctor Consultation,23:12:00,3,General Emergency,Medium,Evening,DR006,0.966667,0.848611,0,2026-01-01 02:50:00,Triage,2026-01-01 01:17:00,93.0
3,CF00001,Treatment,01:02:00,4,General Emergency,Medium,Evening,DR006,0.043056,0.848611,1,2026-01-01 04:40:00,Doctor Consultation,2026-01-01 02:50:00,110.0
4,CF00001,Discharge,01:51:00,5,General Emergency,Medium,Evening,DR006,0.077083,0.848611,1,2026-01-01 05:29:00,Treatment,2026-01-01 04:40:00,49.0


In [7]:
import pandas as pd
import pm4py

df = pd.read_excel("../output of pandas/careflow_cleaned_events.xlsx")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
display(df.head())

Shape: (10485, 15)

Columns: ['Case_ID', 'Activity_Name', 'Timestamp', 'Event_Order', 'Department', 'Triage_Priority', 'Shift', 'Doctor_ID', 'Event_Time', 'Arrival_Time', 'Day_Offset', 'Timestamp_Reconstructed', 'Previous_Activity', 'Previous_Time', 'Transition_Min']

Data types:
Case_ID                            object
Activity_Name                      object
Timestamp                          object
Event_Order                         int64
Department                         object
Triage_Priority                    object
Shift                              object
Doctor_ID                          object
Event_Time                        float64
Arrival_Time                      float64
Day_Offset                          int64
Timestamp_Reconstructed    datetime64[ns]
Previous_Activity                  object
Previous_Time              datetime64[ns]
Transition_Min                    float64
dtype: object


,Case_ID,Activity_Name,Timestamp,Event_Order,Department,Triage_Priority,Shift,Doctor_ID,Event_Time,Arrival_Time,Day_Offset,Timestamp_Reconstructed,Previous_Activity,Previous_Time,Transition_Min
0,CF00001,Registration,20:22:00,1,General Emergency,Medium,Evening,DR006,0.848611,0.848611,0,2026-01-01 00:00:00,NaN,NaT,NaN
1,CF00001,Triage,21:39:00,2,General Emergency,Medium,Evening,DR006,0.902083,0.848611,0,2026-01-01 01:17:00,Registration,2026-01-01 00:00:00,77.0
2,CF00001,Doctor Consultation,23:12:00,3,General Emergency,Medium,Evening,DR006,0.966667,0.848611,0,2026-01-01 02:50:00,Triage,2026-01-01 01:17:00,93.0
3,CF00001,Treatment,01:02:00,4,General Emergency,Medium,Evening,DR006,0.043056,0.848611,1,2026-01-01 04:40:00,Doctor Consultation,2026-01-01 02:50:00,110.0
4,CF00001,Discharge,01:51:00,5,General Emergency,Medium,Evening,DR006,0.077083,0.848611,1,2026-01-01 05:29:00,Treatment,2026-01-01 04:40:00,49.0


In [8]:
import pandas as pd
import pm4py

# Select PM4Py-required columns
event_log_df = df[
    ["Case_ID", "Activity_Name", "Timestamp_Reconstructed"]
].copy()

# Rename columns to PM4Py standard names
event_log_df = event_log_df.rename(columns={
    "Case_ID": "case:concept:name",
    "Activity_Name": "concept:name",
    "Timestamp_Reconstructed": "time:timestamp"
})

# Ensure correct data types and remove invalid required rows
event_log_df["time:timestamp"] = pd.to_datetime(
    event_log_df["time:timestamp"], errors="coerce"
)

event_log_df = event_log_df.dropna(
    subset=["case:concept:name", "concept:name", "time:timestamp"]
)

# Sort events chronologically within each case
event_log_df = event_log_df.sort_values(
    ["case:concept:name", "time:timestamp"]
).reset_index(drop=True)

# Convert DataFrame to PM4Py event log
event_log = pm4py.convert_to_event_log(event_log_df)

print("Cases:", event_log_df["case:concept:name"].nunique())
print("Events:", len(event_log_df))
print("Activities:", event_log_df["concept:name"].nunique())
print("PM4Py event log created successfully!")

Cases: 1500
Events: 10485
Activities: 8
PM4Py event log created successfully!


In [9]:
# Discover a process model using the Inductive Miner
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(
    event_log
)

# Display the process map
pm4py.view_petri_net(net, initial_marking, final_marking)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH